In [1]:
from __future__ import annotations


import re
import sys
import matplotlib.pyplot as plt
import traceback
from dataclasses import dataclass
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import mne

C:\Users\ajars\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# ============================
# CONFIG
# ============================
BIDS_ROOT = Path(r"D:\ds003498")
SESSIONS = ["interictalsleep"]

# Preprocessing params
MAINS_HZ = 50.0                
NOTCH_MAX_HZ = 200.0           
NOTCH_METHOD = "spectrum_fit"   # supports multiple harmonics nicely
REF_MODE = "average"            # CAR

EPOCH_LEN = 5.0
ARTIFACT_MODE = "rms_z"
ARTIFACT_ZTHRESH = 5.0

# HFO band definitions (for saving band-passed epochs)
RIPPLE_BAND = (80, 250)
FAST_RIPPLE_BAND = (250, 500)

# Output derivative name
DERIV_NAME = "preproc_ds003498"

# Robustness
MIN_DURATION_SEC = 5.0

In [3]:
# ============================
# HELPERS
# ============================
@dataclass(frozen=True)
class RunPaths:
    subject: str
    session: str
    run: str
    ieeg_dir: Path
    vhdr: Path
    channels_tsv: Path
    events_tsv: Path
    out_dir: Path
    ripple_out: Path
    fast_out: Path
    stats_out: Path


def parse_bids_from_vhdr(vhdr_path: Path) -> tuple[str, str, str]:
    """
    Extract subject, session, run from a filename like:
      sub-03_ses-interictalsleep_run-01_ieeg.vhdr
    """
    name = vhdr_path.name
    m = re.match(r"sub-(?P<sub>\d+)_ses-(?P<ses>[^_]+)_run-(?P<run>\d+)_ieeg\.vhdr$", name)
    if not m:
        raise ValueError(f"Not a BIDS run file: {vhdr_path}")
    return m.group("sub"), m.group("ses"), m.group("run")


def build_run_paths(vhdr_path: Path) -> RunPaths:
    sub, ses, run = parse_bids_from_vhdr(vhdr_path)

    ieeg_dir = vhdr_path.parent
    bids_root = ieeg_dir.parents[3]  # .../<BIDS_ROOT>/sub-XX/ses-YY/ieeg/<file>

    channels_tsv = ieeg_dir / f"sub-{sub}_ses-{ses}_run-{run}_channels.tsv"
    events_tsv = ieeg_dir / f"sub-{sub}_ses-{ses}_run-{run}_events.tsv"

    out_dir = bids_root / "derivatives" / DERIV_NAME / f"sub-{sub}" / f"ses-{ses}"
    out_dir.mkdir(parents=True, exist_ok=True)

    ripple_out = out_dir / f"sub-{sub}_ses-{ses}_run-{run}_ripple_epo.fif"
    fast_out = out_dir / f"sub-{sub}_ses-{ses}_run-{run}_fast_epo.fif"
    stats_out = out_dir / f"sub-{sub}_ses-{ses}_run-{run}_hfo_annotation_stats.csv"

    return RunPaths(
        subject=sub,
        session=ses,
        run=run,
        ieeg_dir=ieeg_dir,
        vhdr=vhdr_path,
        channels_tsv=channels_tsv,
        events_tsv=events_tsv,
        out_dir=out_dir,
        ripple_out=ripple_out,
        fast_out=fast_out,
        stats_out=stats_out,
    )


def already_processed(p: RunPaths) -> bool:
    # Skip if main epoch outputs exist (same logic as your directory shows)
    return p.ripple_out.exists() and p.fast_out.exists()


def apply_channels_tsv(raw: mne.io.BaseRaw, channels_tsv: Path) -> None:
    if not channels_tsv.exists():
        print(f"  [WARN] Missing channels.tsv: {channels_tsv.name} (skipping bad-channel removal/types)")
        return

    ch = pd.read_csv(channels_tsv, sep="\t")

    # Mark/remove bad channels
    bads = []
    if "status" in ch.columns and "name" in ch.columns:
        bads = ch.loc[ch["status"].astype(str).str.lower() == "bad", "name"].tolist()
        bads = [b for b in bads if b in raw.ch_names]
        raw.info["bads"] = bads
        if bads:
            raw.drop_channels(bads)
    print(f"  Bad channels removed: {len(bads)}")


def load_events_as_annotations(raw: mne.io.BaseRaw, events_tsv: Path) -> None:
    if not events_tsv.exists():
        print(f"  [WARN] Missing events.tsv: {events_tsv.name} (continuing without annotations)")
        return

    ev = pd.read_csv(events_tsv, sep="\t")
    if not {"onset", "duration"}.issubset(ev.columns):
        print(f"  [WARN] events.tsv missing onset/duration columns (continuing without annotations)")
        return

    desc_col = "trial_type" if "trial_type" in ev.columns else ("value" if "value" in ev.columns else None)
    descriptions = ev[desc_col].astype(str).tolist() if desc_col else ["event"] * len(ev)

    max_t = float(raw.times[-1])
    onsets = ev["onset"].astype(float).to_numpy()
    durs = ev["duration"].astype(float).to_numpy()

    # Clip durations to file range (prevents warnings)
    durs = np.clip(durs, 0, np.maximum(0, max_t - onsets))

    raw.set_annotations(mne.Annotations(onset=onsets, duration=durs, description=descriptions))
    print(f"  Annotations set: {len(raw.annotations)}")


def reject_artifacts_rms_z(epochs: mne.Epochs, z_thresh: float) -> tuple[mne.Epochs, np.ndarray]:
    data = epochs.get_data()  # (n_epochs, n_channels, n_times)
    rms = np.sqrt(np.mean(data ** 2, axis=2))  # (n_epochs, n_channels)

    # z-score per channel across epochs
    z = (rms - rms.mean(axis=0, keepdims=True)) / (rms.std(axis=0, keepdims=True) + 1e-12)
    bad_epochs = np.any(np.abs(z) > z_thresh, axis=1)

    return epochs[~bad_epochs], bad_epochs


def parse_hfo_desc(desc: str) -> tuple[str | None, str | None]:
    # Expected: ripple_<ch>, fr_<ch>, frandr_<ch>
    if "_" not in desc:
        return None, None
    kind, ch_name = desc.split("_", 1)
    kind = kind.lower().strip()
    ch_name = ch_name.strip()
    if kind not in {"ripple", "fr", "frandr"}:
        return None, None
    return kind, ch_name


def export_annotation_stats(raw: mne.io.BaseRaw, stats_out: Path) -> None:
    hfo_counts = defaultdict(Counter)

    for ann in raw.annotations:
        kind, ch_name = parse_hfo_desc(str(ann["description"]))
        if kind is None or ch_name is None:
            continue
        hfo_counts[ch_name][kind] += 1

    duration_min = float(raw.times[-1]) / 60.0 if raw.times.size else np.nan

    rows = []
    for ch_name, ctr in hfo_counts.items():
        rows.append(
            {
                "channel": ch_name,
                "ripple_count": ctr.get("ripple", 0),
                "fr_count": ctr.get("fr", 0),
                "frandr_count": ctr.get("frandr", 0),
                "duration_min": duration_min,
                "ripple_rate_per_min": ctr.get("ripple", 0) / duration_min if duration_min and duration_min > 0 else np.nan,
                "fr_rate_per_min": ctr.get("fr", 0) / duration_min if duration_min and duration_min > 0 else np.nan,
                "frandr_rate_per_min": ctr.get("frandr", 0) / duration_min if duration_min and duration_min > 0 else np.nan,
            }
        )

    hfo_df = pd.DataFrame(rows)
    if not hfo_df.empty and "frandr_rate_per_min" in hfo_df.columns:
        hfo_df = hfo_df.sort_values("frandr_rate_per_min", ascending=False)

    hfo_df.to_csv(stats_out, index=False)


def notch_harmonics(raw: mne.io.BaseRaw, mains_hz: float, max_hz: float) -> None:
    """
    Apply notch at mains_hz and its harmonics up to max_hz using spectrum_fit.
    This is applied on the broadband signal BEFORE bandpass to ripple/fast ripple.
    """
    if mains_hz <= 0:
        return
    n_harm = int(max_hz // mains_hz)
    if n_harm < 1:
        return
    freqs = [mains_hz * k for k in range(1, n_harm + 1)]
    raw.notch_filter(freqs=freqs, method=NOTCH_METHOD, verbose=False)

In [4]:
def preprocess_one_run(p: RunPaths) -> None:
    print(f"\n=== sub-{p.subject} ses-{p.session} run-{p.run} ===")

    if already_processed(p):
        print("  [SKIP] Already preprocessed (epoch files exist).")
        return

    if not p.vhdr.exists():
        print("  [SKIP] Missing vhdr.")
        return

    # 1) Load raw (guard broken headers / empty files)
    try:
        raw = mne.io.read_raw_brainvision(str(p.vhdr), preload=True, verbose=False)
    except Exception as e:
        print("  [SKIP] Failed to read BrainVision:", repr(e))
        return

    if raw.times.size == 0 or raw.n_times == 0:
        print("  [SKIP] Empty signal (n_times=0).")
        return

    dur = float(raw.times[-1])
    print(f"  Loaded raw | sfreq={raw.info['sfreq']} Hz | ch={len(raw.ch_names)} | dur={dur:.1f}s")

    if dur < MIN_DURATION_SEC:
        print(f"  [SKIP] Too short for epoching (<{MIN_DURATION_SEC}s).")
        return

    # 2) channels.tsv (drop bad channels)
    apply_channels_tsv(raw, p.channels_tsv)

    # 3) Load HFO annotations (if present)
    load_events_as_annotations(raw, p.events_tsv)

    # 4) Notch (50Hz + harmonics up to NOTCH_MAX_HZ) + CAR
    notch_harmonics(raw, mains_hz=MAINS_HZ, max_hz=NOTCH_MAX_HZ)
    raw.set_eeg_reference(ref_channels=REF_MODE, verbose=False)

    # 5) Bandpass into ripple / fast ripple
    raw_ripple = raw.copy().filter(l_freq=RIPPLE_BAND[0], h_freq=RIPPLE_BAND[1], phase="zero", verbose=False)
    raw_fast = raw.copy().filter(l_freq=FAST_RIPPLE_BAND[0], h_freq=FAST_RIPPLE_BAND[1], phase="zero", verbose=False)

    # 6) Epoch into fixed segments
    try:
        events = mne.make_fixed_length_events(raw_ripple, duration=EPOCH_LEN)
    except ValueError as e:
        print("  [SKIP] No events produced for epoching:", repr(e))
        return

    if events.size == 0:
        print("  [SKIP] No events produced (events empty).")
        return

    epochs_ripple = mne.Epochs(raw_ripple, events, tmin=0, tmax=EPOCH_LEN, baseline=None, preload=True, verbose=False)
    epochs_fast = mne.Epochs(raw_fast, events, tmin=0, tmax=EPOCH_LEN, baseline=None, preload=True, verbose=False)
    print(f"  Epochs: ripple={len(epochs_ripple)} | fast={len(epochs_fast)} | len={EPOCH_LEN}s")

    # 7) Artifact rejection
    if ARTIFACT_MODE == "rms_z":
        epochs_ripple_clean, bad_rip = reject_artifacts_rms_z(epochs_ripple, z_thresh=ARTIFACT_ZTHRESH)
        epochs_fast_clean, bad_fast = reject_artifacts_rms_z(epochs_fast, z_thresh=ARTIFACT_ZTHRESH)
        print(
            f"  Rejected epochs: ripple={int(bad_rip.sum())}/{len(epochs_ripple)} | "
            f"fast={int(bad_fast.sum())}/{len(epochs_fast)}"
        )
    else:
        epochs_ripple_clean, epochs_fast_clean = epochs_ripple, epochs_fast

    # 8) Export annotation stats
    export_annotation_stats(raw, p.stats_out)
    print(f"  Saved stats: {p.stats_out.name}")

    # 9) Save clean epochs
    epochs_ripple_clean.save(str(p.ripple_out), overwrite=True)
    epochs_fast_clean.save(str(p.fast_out), overwrite=True)
    print(f"  Saved epochs: {p.ripple_out.name} | {p.fast_out.name}")


def find_all_vhdr(bids_root: Path, sessions: list[str]) -> list[Path]:
    vhdrs: list[Path] = []
    for ses in sessions:
        vhdrs.extend(bids_root.glob(f"sub-*/ses-{ses}/ieeg/sub-*_ses-{ses}_run-*_ieeg.vhdr"))
    return sorted(vhdrs)

In [5]:
def main() -> int:
    if not BIDS_ROOT.exists():
        print(f"[ERROR] BIDS_ROOT does not exist: {BIDS_ROOT}")
        return 2

    vhdrs = find_all_vhdr(BIDS_ROOT, SESSIONS)
    if not vhdrs:
        print("[ERROR] No .vhdr runs found. Check BIDS_ROOT and SESSIONS.")
        return 2

    print(f"Found {len(vhdrs)} runs (.vhdr) across sessions={SESSIONS}")
    print(f"Derivative output root: {BIDS_ROOT / 'derivatives' / DERIV_NAME}")
    print(f"Notch: {MAINS_HZ}Hz harmonics up to {NOTCH_MAX_HZ}Hz using {NOTCH_METHOD} | CAR={REF_MODE}")

    n_ok = 0
    n_skip = 0
    n_fail = 0

    for vhdr in vhdrs:
        try:
            rp = build_run_paths(vhdr)
            if already_processed(rp):
                n_skip += 1
                continue

            preprocess_one_run(rp)
            # count as processed only if outputs now exist
            if already_processed(rp):
                n_ok += 1
            else:
                # preprocess_one_run can "skip" internally; treat as skip
                n_skip += 1
        except Exception as e:
            n_fail += 1
            print(f"\n[FAIL] {vhdr}")
            print("Reason:", repr(e))
            traceback.print_exc()

    print("\n====================")
    print("DONE")
    print(f"Processed: {n_ok}")
    print(f"Skipped:   {n_skip}")
    print(f"Failed:    {n_fail}")
    return 0 if n_fail == 0 else 1


if __name__ == "__main__":
    raise SystemExit(main())

Found 385 runs (.vhdr) across sessions=['interictalsleep']
Derivative output root: D:\ds003498\derivatives\preproc_ds003498
Notch: 50.0Hz harmonics up to 200.0Hz using spectrum_fit | CAR=average

=== sub-19 ses-interictalsleep run-04 ===
  [SKIP] Failed to read BrainVision: IndexError('index -1 is out of bounds for axis 0 with size 0')

=== sub-19 ses-interictalsleep run-05 ===
  [SKIP] Failed to read BrainVision: No section: 'Common infos'

=== sub-20 ses-interictalsleep run-06 ===


C:\Users\ajars\AppData\Local\Temp\ipykernel_14200\2140133907.py:14: RuntimeWarning: MNE-Python currently only supports header versions 1.0 and 2.0, got unparsable '/annex/objects/MD5E-s1587--a860d4c80725bd390548f1bbcfa24212.vhdr'. Contact MNE-Python developers for support.
  raw = mne.io.read_raw_brainvision(str(p.vhdr), preload=True, verbose=False)


  Loaded raw | sfreq=2000.0 Hz | ch=16 | dur=0.0s
  [SKIP] Too short for epoching (<5.0s).

DONE
Processed: 0
Skipped:   385
Failed:    0


C:\Users\ajars\AppData\Local\Temp\ipykernel_14200\2140133907.py:14: RuntimeWarning: Omitted 7222 annotation(s) that were outside data range.
  raw = mne.io.read_raw_brainvision(str(p.vhdr), preload=True, verbose=False)


SystemExit: 0

C:\Users\ajars\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
